# Speech Emotion Recognition on RAVDESS — Project Walkthrough

**ISB AMPBA foundation project · Team of 5**

---

## Read this first

This notebook is the **guided tour** of the project. It is written so that anyone on
the team can read it top to bottom and understand *what we built, what we found, and
why we made each choice* — without having to read the source code.

**Important convention:** this notebook contains **no logic**. Every function it calls
lives in the `ser/` package and is covered by tests. The notebook only *calls* and
*narrates*. That matters because:

- Logic in a notebook cannot be unit-tested, and cannot be reused by four other people.
- If two of us copy-paste the same cell and one edits it, our numbers silently diverge.

So: **`ser/` is the code. Notebooks are the story.**

---

## What we are actually predicting

Given ~3 seconds of speech audio, predict which of **8 emotions** the speaker was
performing: `neutral, calm, happy, sad, angry, fearful, disgust, surprised`.

**Business framing:** contact-centre / telecaller quality assurance and CSAT
prediction. If we can detect frustration or anger in a call, we can flag it for review.

**Dataset:** RAVDESS — 1,440 `.wav` clips, 24 professional actors, each acting all 8
emotions from a fixed script.

---

## The one thing to understand about this project

Actors are the trap. If the same actor's voice appears in both training and test data,
the model can learn *"this is Actor 7, and Actor 7 sounds like this when angry"* instead
of learning what anger sounds like in general. It then scores brilliantly — and fails
completely on a real telecaller it has never heard.

**Almost every published RAVDESS result you will find online has this flaw.**

We measure it deliberately. Everything below is built around that.

> ## ⚠ These are the FLOOR numbers, not the final ones
>
> This notebook walks through the project's construction, and its results section
> reports the **untuned floor: 51.5% accuracy / 49.8% macro-F1.** That was correct
> when written and is kept deliberately, because the progression is part of the story.
>
> **The final numbers are 63.9% accuracy / 63.3% macro-F1** — see
> `01_results_and_findings.ipynb`. Feature engineering (Phase 4) added +13.31 points;
> hyperparameter tuning added +0.22.
>
> Read this notebook to understand *how the project is built*. Read `01` for *what it found*.

## Setup

We add the repo root to the path so `import ser` works from the `notebooks/` folder.

In [1]:
import sys
from pathlib import Path

# Make the repo root importable, so `from ser import ...` works from notebooks/.
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

print("repo root:", REPO)

repo root: C:\Users\kajas\OneDrive\Desktop\Unheard


---
# Part 0 — The rules we work under

These are in `CLAUDE.md` and they are **non-negotiable**. They exist because five
people are producing numbers that have to be comparable to each other.

| # | Rule | Why it exists |
|---|---|---|
| 1 | **Never modify `splits/folds.csv`** | It is the contract. If your folds differ from mine, our accuracies are not comparable — and neither of us would notice. |
| 2 | **Never fit anything on test data** | Scalers, PCA, feature selection, class weights. Fit on train, apply to test. Anything else inflates the score. |
| 3 | **Augmentation on training folds only** | Augmented test data measures nothing real. |
| 4 | **Speaker-independent is primary** | Random-split numbers are only ever shown *beside* it, as the leakage comparison. Never alone. |
| 5 | **Every run appends to `results/results.csv`** | A result that is not logged does not exist. |
| 6 | **Fix all seeds** (`SEED = 42`) | So a rerun reproduces the number. |
| 7 | **Never commit audio or features** | `data/` and `features/` are gitignored. Audio is huge; features are regenerable. |
| 8 | **Diverge from the plan → stop and re-plan** | Prevents scope drift. |

Rules 1, 2 and 4 are the ones that protect the *validity* of our results. The rest
protect our *sanity*.

---
# Part 1 — The data, and three surprises

`ser/metadata.py` turns filenames into a table. It never opens an audio file — it only
reads names and folder structure. That makes it fast and makes it the single source of
truth for *what clips exist*.

In [2]:
from ser.metadata import load

meta = load()
print(f"{len(meta)} clips, {meta.shape[1]} columns")
meta.head(3)

1440 clips, 14 columns


,filename,filepath,modality,vocal_channel,emotion,intensity,statement,repetition,actor,emotion_label,intensity_label,statement_label,repetition_label,gender
0,01-01-01-01-01.wav,C:\Users\kajas\OneDrive\Desktop\Unheard\data\d...,03,01,01,01,01,01,1,neutral,normal,kids,1st,male
1,01-01-01-02-01.wav,C:\Users\kajas\OneDrive\Desktop\Unheard\data\d...,03,01,01,01,01,02,1,neutral,normal,kids,2nd,male
2,01-01-02-01-01.wav,C:\Users\kajas\OneDrive\Desktop\Unheard\data\d...,03,01,01,01,02,01,1,neutral,normal,dogs,1st,male


### How a filename decodes

A RAVDESS filename is a set of hyphen-separated codes. Ours look like
`06-02-01-02-12.wav`:

| Position | Field | Example | Meaning |
|---|---|---|---|
| 1 | emotion | `06` | fearful |
| 2 | intensity | `02` | strong |
| 3 | statement | `01` | "Kids are talking by the door" |
| 4 | repetition | `02` | 2nd take |
| 5 | actor | `12` | Actor 12 (even ⇒ female) |

`load()` expands those codes into readable labels and derives `gender` from the actor
number — **odd = male, even = female**, which is the RAVDESS convention.

In [3]:
# Every count below is ASSERTED in tests/test_metadata.py, not assumed.
print("Clips per emotion:")
print(meta["emotion_label"].value_counts().to_string())
print()
print("Actors per gender:", meta.groupby("gender")["actor"].nunique().to_dict())
print("Clips per actor   :", meta["actor"].value_counts().unique(), "(all identical)")

Clips per emotion:
emotion_label
calm         192
happy        192
sad          192
angry        192
fearful      192
disgust      192
surprised    192
neutral       96

Actors per gender: {'female': 12, 'male': 12}
Clips per actor   : [60] (all identical)


### Surprise 1 — neutral has half the clips

Notice `neutral = 96` while every other emotion has `192`. This is **not** a broken
download. RAVDESS never recorded neutral at *strong* intensity — you cannot perform
"very neutral". So neutral has only the normal-intensity half.

**Consequence for us:** accuracy alone will hide failure on neutral, because it is only
6.7% of the data. This is exactly why we report **macro-F1** alongside accuracy —
macro-F1 gives every class equal weight regardless of size.

In [4]:
# Proof: there is no strong-intensity neutral, and the other 7 split evenly.
crosstab = pd.crosstab(meta["emotion_label"], meta["intensity_label"])
crosstab

intensity_label,normal,strong
emotion_label,,
angry,96,96
calm,96,96
disgust,96,96
fearful,96,96
happy,96,96
neutral,96,0
sad,96,96
surprised,96,96


### Surprise 2 — our filenames have 5 fields, not the canonical 7

Standard RAVDESS filenames have **seven** fields, starting with `modality` (`03` =
audio-only) and `vocal-channel` (`01` = speech). Both are constant for the speech
subset, and whoever repackaged our copy **stripped them**.

Our parser handles this by being **strict rather than clever**: it accepts exactly 5
fields and *rejects* a 7-field name outright.

**Why reject instead of auto-detecting?** If a canonical 7-field file were dropped into
our data folder and the parser tried to be helpful, every field would shift by two
positions — `modality` would be read as `emotion`. Every clip would be silently
mislabelled and the error would only surface as an inexplicably bad model. A loud crash
is far better than quiet corruption.

In [5]:
from ser.metadata import parse_filename

# The canonical 7-field name is rejected, deliberately and loudly.
try:
    parse_filename("03-01-06-01-02-01-12.wav")
except ValueError as exc:
    print("REJECTED as intended:\n ", exc)

REJECTED as intended:
  '03-01-06-01-02-01-12.wav': expected 5 hyphen-separated fields (emotion-intensity-statement-repetition-actor_code), got 7. Canonical seven-field RAVDESS names are not accepted here -- see module docstring.


### Surprise 3 — the folder layout is not what the spec said

Our copy stores clips in `data/data/PersonNN/` — note the **doubled `data`** and
`Person` rather than the canonical `Actor_`. We found this by looking, not by assuming.

The actor is therefore encoded **twice**: in the folder name *and* in filename field 5.
We check they agree in all 1,440 files — a cheap, high-value integrity check.

In [6]:
from_field = meta["filename"].str.split("-").str[4].str.replace(".wav", "", regex=False)
from_folder = meta["filepath"].str.extract(r"Person(\d{2})")[0]

print("Files where folder and filename disagree:", int((from_field != from_folder).sum()))
print("(0 means the dataset is internally consistent)")

Files where folder and filename disagree: 0
(0 means the dataset is internally consistent)


---
# Part 2 — The audio audit, and the finding that changed our plan

`ser/audit.py` opens all 1,440 files and measures them. Full write-up:
**`docs/data_audit.md`**.

**Crucial detail:** the audit loads at the **native 48 kHz** (`sr=None`). This matters
because `librosa.load()` *silently resamples to 22,050 Hz* if you do not pass `sr=`.
An audit run at the default would describe a resampled artifact, not our actual files.

In [7]:
from ser.preprocess import load_audio, channel_report, TARGET_SR, NATIVE_SR
from ser.metadata import DATA_ROOT

clip = DATA_ROOT / "Person01" / "01-01-01-01-01.wav"

y_native, sr_native = load_audio(clip, sr=None)        # true file
y_target, sr_target = load_audio(clip)                 # project default

print(f"native rate  : {sr_native} Hz, {len(y_native):,} samples, {y_native.dtype}")
print(f"project rate : {sr_target} Hz, {len(y_target):,} samples  <- TARGET_SR")
print(f"duration is preserved: {len(y_native)/sr_native:.3f}s vs {len(y_target)/sr_target:.3f}s")

C:\Users\kajas\OneDrive\Desktop\Unheard\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


native rate  : 48000 Hz, 158,558 samples, float32
project rate : 16000 Hz, 52,853 samples  <- TARGET_SR
duration is preserved: 3.303s vs 3.303s


### `load_audio` is the ONLY way we open audio

Never call `librosa.load` directly. `ser.preprocess.load_audio` is the single entry
point, and it enforces two project-wide decisions in one place:

1. **Mono downmix** — 5 of our 1,440 files are secretly dual-channel.
2. **16 kHz target rate** — and never the silent 22,050 default.

Having one function is *why* those decisions actually hold everywhere, instead of being
re-decided (or forgotten) at each call site.

In [8]:
# The 5 dual-channel files have BIT-IDENTICAL channels, so averaging is lossless.
stereo_clip = DATA_ROOT / "Person01" / "02-01-01-02-01.wav"
report = channel_report(stereo_clip)

print("channels stored in the file :", report["n_channels"])
print("largest difference L vs R   :", report["max_channel_diff"], "<- exactly 0.0")
print()
print("So (L+R)/2 returns the original signal exactly. No information is lost,")
print("and we never modify the raw files on disk.")

channels stored in the file : 2
largest difference L vs R   : 0.0 <- exactly 0.0

So (L+R)/2 returns the original signal exactly. No information is lost,
and we never modify the raw files on disk.


### What the audit found

| Property | Result |
|---|---|
| Corrupt / zero-length files | **0** — the corpus is clean |
| Sample rate | 48,000 Hz, uniform across all 1,440 |
| Encoding | WAV PCM_16, uniform |
| Duration | 2.94 – 5.27 s (median 3.67) |
| **Silence per clip** | **mean 50.6%**, range 0.4% – 64.6% |
| Peak amplitude | 0.006 – 0.999 (~44 dB spread) |
| Clipped samples | 1 sample, in 1 file — negligible |

**Half of the average clip is silence.** That is the single biggest preprocessing
finding: raw clip duration measures how long the engineer left the recorder running,
not how long the actor spoke.

### ⚠ The finding that reversed our plan

Here is the part worth explaining to anyone reviewing this project, because it is a
genuine result rather than a settings choice.

The reflexive thing to do with audio is **normalise every clip to the same peak
volume**. It is the default in almost every tutorial. We were about to do it.

Then we measured how loudness actually varies:

In [9]:
audit = pd.read_parquet(REPO / "features" / "audit.parquet")

by_emotion = audit.pivot_table(
    index="emotion_label", columns="intensity_label", values="peak_amp", aggfunc="mean"
).round(3).sort_values("strong", ascending=False)

by_actor = audit.groupby("actor")["rms"].mean()

print("Mean PEAK AMPLITUDE by emotion and intensity:")
print(by_emotion.to_string())
print()
print(f"Loudest emotion / quietest emotion : {by_emotion.max().max() / by_emotion.min().min():.1f}x")
print(f"Loudest actor   / quietest actor   : {by_actor.max() / by_actor.min():.1f}x")

Mean PEAK AMPLITUDE by emotion and intensity:
intensity_label  normal  strong
emotion_label                  
angry             0.185   0.750
fearful           0.107   0.430
happy             0.098   0.304
surprised         0.098   0.172
disgust           0.080   0.145
sad               0.048   0.123
calm              0.041   0.039
neutral           0.054     NaN

Loudest emotion / quietest emotion : 19.2x
Loudest actor   / quietest actor   : 2.9x


**Read those last two numbers carefully.**

- Emotion drives loudness by roughly **19×** (angry-strong vs calm-strong, ~26 dB).
- Speaker/recording gain drives it by only **2.9×** (~9 dB).

So loudness is overwhelmingly carrying **emotion**, not recording conditions. Per-clip
peak normalisation would have **deleted the single most discriminative cue in the
dataset** in order to suppress a much smaller nuisance.

**Decision: we do NOT peak-normalise.** The smaller per-speaker gain component is
handled at the *feature* level instead — per-speaker z-scoring is a Phase 4 experiment
where we can measure whether it helps, rather than assuming it here.

This is guarded by a test (`test_loudness_is_driven_by_emotion_not_by_actor`) so nobody
can quietly reintroduce normalisation later.

> **Takeaway for the deck:** "we measured instead of assuming, and the measurement
> contradicted the textbook default."

---
# Part 3 — The splits (the most important part)

This is the machinery that makes our results honest, and it is frozen in
`splits/folds.csv`. Full write-up: **`docs/splits.md`**.

## Why we changed the phase order

`PLAN.md` originally had EDA (Phase 2) before splits (Phase 3). **We swapped them.**

Reason: EDA means looking at the data. If we explore all 1,440 clips before the folds
exist, we are looking at our own test set. Even with no model trained, that leaks
*through us* — every later choice about which features to try and which confusions to
chase is informed by test data. It is unquantifiable and impossible to undo.

So: build the folds first, then restrict EDA to training folds.

## The four protocols

In [10]:
from ser.splits import load_folds, summarise_folds, get_split, PROTOCOLS

folds = load_folds()
print("Protocols available:", PROTOCOLS)
print(f"folds.csv holds {len(folds):,} rows")
folds.head(3)

Protocols available: ('speaker_independent', 'random_stratified', 'statement_holdout', 'statement_holdout_si')
folds.csv holds 19,440 rows


,protocol,fold,filename,split
0,speaker_independent,0,01-01-01-01-04.wav,test
1,speaker_independent,0,01-01-01-01-09.wav,test
2,speaker_independent,0,01-01-01-01-14.wav,test


In [11]:
summarise_folds(folds, meta)

,protocol,fold,n_train,n_test,train_actors,test_actors,actor_overlap,test_pct_male,test_neutral,test_emotions
0,speaker_independent,0,1140,300,19,5,0,40.0,20,8
1,speaker_independent,1,1140,300,19,5,0,60.0,20,8
2,speaker_independent,2,1140,300,19,5,0,40.0,20,8
3,speaker_independent,3,1140,300,19,5,0,60.0,20,8
4,speaker_independent,4,1200,240,20,4,0,50.0,16,8
5,random_stratified,0,1152,288,24,24,24,47.6,20,8
6,random_stratified,1,1152,288,24,24,24,54.9,19,8
7,random_stratified,2,1152,288,24,24,24,47.9,19,8
8,random_stratified,3,1152,288,24,24,24,49.0,19,8
9,random_stratified,4,1152,288,24,24,24,50.7,19,8


### Reading that table

**`speaker_independent`** — our **primary** protocol. `actor_overlap = 0` on every
fold: no actor is ever in both train and test. This is the number we report.

**`random_stratified`** — `actor_overlap = 24` on every fold. **Every actor is on both
sides.** That is not a bug; it is the entire point. This protocol exists *only* to
measure how much a naive random split inflates the score. We never report it alone.

**`statement_holdout`** — train on statement 01, test on statement 02. Does emotion
recognition survive a change of sentence? Note `actor_overlap = 24` here too, so a good
score is ambiguous.

**`statement_holdout_si`** — the fix for that ambiguity. Speakers **and** sentences both
held out (`actor_overlap = 0`). Comparing this against `statement_holdout` separates
"generalises across words" from "recognises the voice".

### Why the folds are unequal — and why we accepted it

Look at `speaker_independent` fold 4: **240 clips over 4 actors**, while folds 0–3 have
300 clips over 5 actors. And gender runs 40/60 rather than 50/50.

This is not sloppiness. **24 actors do not divide by 5.** You get 5,5,5,5,4 — and a
5-actor fold has odd parity, so it can *never* be 50/50 on gender.

We tested the alternatives before accepting this:

| Option | Result |
|---|---|
| Hand-balanced k=5 | **Identical** 20 pp gender gap. No improvement possible. |
| `GroupKFold` k=6 or k=4 | **Catastrophic** — produces single-gender folds (0% or 100% male) |
| Hand-assigned k=6 | Genuinely perfect (2M+2F per fold) — but deviates from the agreed plan |

The k=6 result is a genuine trap worth knowing about: it *looks* like the obvious fix
because 24 divides evenly by 6, but sklearn's assignment happens to group same-gender
actors together, which would have silently wrecked our fairness analysis.

**Consequence:** because fold 4 is 20% smaller, averaging five per-fold accuracies
over-weights each of its clips. So we **pool** instead (next section).

### How to use the folds in your own experiment

Two lines. Do not build your own splits.

In [12]:
train_files, test_files = get_split("speaker_independent", fold=0)

print(f"train: {len(train_files)} clips")
print(f"test : {len(test_files)} clips")
print("overlap between them:", len(set(train_files) & set(test_files)), "<- must be 0")
print()
print("first 2 test files:", test_files[:2])

train: 1140 clips
test : 300 clips
overlap between them: 0 <- must be 0

first 2 test files: ['01-01-01-01-04.wav', '01-01-01-01-09.wav']


Note these are **filenames, not row numbers**. That is deliberate: a list of positional
indices means something completely different if applied to a DataFrame in a different
order, and that failure would be invisible. Filenames cannot be misapplied silently.

---
# Part 4 — The scoring harness

`ser/evaluate.py` has **one** scoring function that everyone calls. Nobody computes
accuracy inline in their own notebook.

Why one function: if five people each write their own scoring code, five subtly
different numbers appear and no one can tell whether a difference is the model or the
metric.

Every call appends a row to `results/results.csv`. **A result that is not logged does
not exist.**

In [13]:
from ser.evaluate import evaluate, RESULTS_COLUMNS

print("results.csv schema:")
for col in RESULTS_COLUMNS:
    print("  -", col)

results.csv schema:
  - run_id
  - timestamp
  - owner
  - protocol
  - fold
  - feature_config
  - model
  - hyperparams
  - accuracy
  - macro_f1
  - notes


### Pooled scoring is our headline number

Because the folds are unequal, we do **not** average the five per-fold accuracies.
Instead `evaluate_pooled` concatenates every clip's held-out prediction into one
1,440-row vector and scores that once.

- Every clip counts exactly once, so the small fold cannot be over-weighted.
- The confusion matrix is dense enough for the neutral-class analysis to be stable.

Per-fold rows are **still** logged, so we can report the spread. Quote the pooled
number *with* the per-fold standard deviation beside it.

---
# Part 5 — Features

`ser/features.py` is **parameterised, not hardcoded**. Comparing feature choices *is*
the Phase 4 experiment, so nothing is baked in.

The config we are using as our first baseline:

In [14]:
from ser.features import FeatureConfig, extract, feature_columns

config = FeatureConfig()   # the defaults ARE our chosen baseline config
print("name :", config.name)
print("hash :", config.hash, " -> features/features_%s.parquet" % config.hash)
print()
for field, value in config.__dict__.items():
    print(f"  {field:15s} = {value}")

name : mfcc40-d-dd-mean_std-trim
hash : c24c36b42cee  -> features/features_c24c36b42cee.parquet

  n_mfcc          = 40
  deltas          = True
  delta_deltas    = True
  aggregation     = mean_std
  trim            = True
  top_db          = 30
  sr              = 16000
  n_fft           = 2048
  hop_length      = 512
  delta_width     = None
  n_mels          = 128
  fmin            = 0.0
  fmax            = None
  extras          = ()
  normalisation   = none


### What that config means, in words

- **40 MFCCs** — Mel-Frequency Cepstral Coefficients. A compact description of the
  shape of the vocal tract at each moment; the standard speech feature.
- **+ delta, + delta-delta** — first and second derivatives over time. MFCCs alone
  describe a snapshot; deltas describe *how the voice is changing*, which is much of
  what makes emotion audible.
- **mean + std aggregation** — a clip has hundreds of frames but a classifier needs one
  fixed-length vector. We take the mean and standard deviation of each coefficient over
  time. So `40 × 3 (mfcc, Δ, ΔΔ) × 2 (mean, std) = 240 features`.
- **trimmed** — leading/trailing silence removed first, because half the average clip is
  silence and averaging over it dilutes everything.
- **no per-speaker normalisation** — deliberately off for this floor run; it is a
  Phase 4 experiment.

The cache is keyed by a hash of the config, so an existing config is never recomputed.

In [15]:
features = extract(config, meta)      # loads from cache if already built
cols = feature_columns(features, meta)

print(f"{len(features)} clips x {len(cols)} features")
print("no NaNs:", not features[cols].isna().any().any())
print()
print("first few feature names:", cols[:4], "...", cols[-2:])

1440 clips x 240 features
no NaNs: True

first few feature names: ['mfcc00_mean', 'mfcc01_mean', 'mfcc02_mean', 'mfcc03_mean'] ... ['dd_mfcc38_std', 'dd_mfcc39_std']


### Where leakage would sneak in — and how we stop it

Feature *extraction* is per-clip: nothing about clip A affects clip B, so running it on
test data is safe.

**Scaling is different.** A `StandardScaler` computes a mean and standard deviation
across clips. If you fit it on all 1,440 clips before splitting, test-set statistics
leak into training. That inflates your score and is very easy to do by accident.

Our defence is structural: the scaler is wrapped in an sklearn `Pipeline` with the
model. `Pipeline.fit()` fits the scaler on the training fold only; `Pipeline.predict()`
merely *applies* those already-fitted statistics to test. **There is no code path that
fits on test data.**

In [16]:
from ser.models.classical import build

pipeline = build("svm-rbf")
pipeline    # scaler and estimator bound together, so they cannot be separated

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('estimator', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo random number generation for shuffling the data forprobability estimates. Ignored when `probability` is False.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3


---
# Part 6 — Baseline results

Two models (SVM-RBF, RandomForest) × two protocols × 5 folds, **library defaults, no
tuning**. This is a **floor**, not a result — the number every later model must beat to
justify its complexity.

All 24 runs are already logged in `results/results.csv`. We read them back rather than
recomputing, which is the point of having a results log.

In [17]:
results = pd.read_csv(REPO / "results" / "results.csv")
FLOOR = "mfcc40-d-dd-mean_std-trim"   # this notebook reports the floor
floor_only = results[results["feature_config"] == FLOOR]
per_fold = floor_only[floor_only["fold"] != "pooled"].copy()
per_fold["accuracy"] = per_fold["accuracy"].astype(float)
per_fold["macro_f1"] = per_fold["macro_f1"].astype(float)

summary = (
    per_fold.groupby(["model", "protocol"])
    .agg(
        acc_mean=("accuracy", "mean"),
        acc_sd=("accuracy", "std"),
        acc_min=("accuracy", "min"),
        acc_max=("accuracy", "max"),
        f1_mean=("macro_f1", "mean"),
        f1_sd=("macro_f1", "std"),
    )
    .round(4)
)
summary

acc_mean  acc_sd  acc_min  acc_max  f1_mean   f1_sd
model         protocol                                                                
random-forest random_stratified      0.5854  0.0303   0.5486   0.6215   0.5520  0.0343
              speaker_independent    0.4648  0.0712   0.3733   0.5333   0.4187  0.0698
svm-rbf       random_stratified      0.6521  0.0298   0.6076   0.6875   0.6341  0.0362
              speaker_independent    0.5130  0.0607   0.4667   0.6133   0.4910  0.0696

### The leakage gap — our headline finding

This is the number that justifies the whole speaker-independent apparatus.

In [18]:
pooled = floor_only[floor_only["fold"] == "pooled"].copy()
pooled["accuracy"] = pooled["accuracy"].astype(float)
pooled["macro_f1"] = pooled["macro_f1"].astype(float)

gap = pooled.pivot_table(index="model", columns="protocol",
                         values="accuracy", aggfunc="max")
gap["leakage_gap_pp"] = ((gap["random_stratified"] - gap["speaker_independent"]) * 100).round(2)
gap["inflation_%"] = ((gap["random_stratified"] / gap["speaker_independent"] - 1) * 100).round(1)
gap.round(4)

protocol,random_stratified,speaker_independent,leakage_gap_pp,inflation_%
model,,,,
random-forest,0.5854,0.4674,11.81,25.3
svm-rbf,0.6521,0.5146,13.75,26.7


**Read this as:** a random train/test split makes SVM-RBF look like it scores **65.2%**.
Held out by speaker — which is what deployment actually looks like — it scores
**51.5%**.

**The random split inflates accuracy by 13.75 percentage points, or 26.7% relative.**

That gap *is* the contribution. It is the difference between a number that would look
good in a report and a number that would survive contact with a real telecaller.

### Fold-to-fold variance — always report this

With only 24 actors split into 5 groups, *which* actors land in the test fold matters a
lot. Never quote a speaker-independent accuracy without its spread.

In [19]:
si = per_fold[per_fold["protocol"] == "speaker_independent"]
fold_table = si.pivot_table(index="fold", columns="model",
                            values="accuracy", aggfunc="max").round(4)
fold_table.loc["**sd**"] = fold_table.std().round(4)
fold_table.loc["**range**"] = (fold_table.iloc[:5].max() - fold_table.iloc[:5].min()).round(4)
fold_table

model,random-forest,svm-rbf
fold,,
0,0.3733,0.4667
1,0.5033,0.5267
2,0.5333,0.6133
3,0.5100,0.4833
4,0.4042,0.4750
**sd**,0.0712,0.0607
**range**,0.1600,0.1466


Speaker-independent standard deviation is **~6 pp**, versus **~3 pp** for the random
split — **twice the variance**. That is the expected signature: swapping five whole
actors changes the problem much more than swapping 288 random clips.

Practically: the SVM's 51.5% carries a 95% interval of roughly **±7.5 pp**. It could
reasonably have been 44% or 59%. Anyone quoting "51.5%" as a precise figure is
overstating what 5 folds of 24 actors can tell us.

Fold 2 (SVM, 61.3%) sits about 1.6 standard deviations high. Noticeable, not an outlier
— worth revisiting in Phase 7 to see whether those particular actors are unusually easy.

### Confusion matrix — best speaker-independent run

SVM-RBF, pooled over all 1,440 held-out clips. **Rows = truth, columns = prediction.**
The diagonal is correct answers; everything off-diagonal is a mistake.

In [20]:
from ser.evaluate import evaluate_pooled
from ser.splits import get_split

indexed = features.set_index("filename")
X, y = indexed[cols], indexed["emotion_label"]

per_fold_predictions = []
for fold in range(5):
    train_files, test_files = get_split("speaker_independent", fold, folds=folds)
    model = build("svm-rbf")
    model.fit(X.loc[train_files].to_numpy(), y.loc[train_files].to_numpy())
    predicted = model.predict(X.loc[test_files].to_numpy())
    per_fold_predictions.append(
        (y.loc[test_files].to_numpy(), predicted, indexed.loc[test_files].reset_index())
    )

# append=False: this is a reproduction for the notebook, not a new experiment,
# so it must NOT add a row to the shared results log.
best = evaluate_pooled(
    per_fold_predictions, "speaker_independent",
    model="svm-rbf", feature_config=config.name,
    notes="notebook reproduction", append=False,
)
print(f"accuracy {best['accuracy']:.4f}   macro-F1 {best['macro_f1']:.4f}   n={best['n']}")

accuracy 0.5146   macro-F1 0.4980   n=1440


In [21]:
# Darker cell = more clips. A perfect model would be dark only on the diagonal.
best["confusion_matrix"].style.background_gradient(cmap="Blues", axis=None).format("{:.0f}")

predicted,neutral,calm,happy,sad,angry,fearful,disgust,surprised
true,,,,,,,,
neutral,22,22,20,18,1,3,8,2
calm,10,141,4,23,0,4,10,0
happy,4,9,82,13,19,47,8,10
sad,8,47,21,65,3,23,22,3
angry,3,3,11,7,114,19,26,9
fearful,1,12,34,20,7,108,7,3
disgust,1,18,6,24,19,12,108,4
surprised,3,5,20,13,11,23,16,101


### What the mistakes tell us

Three confusions dominate, and they are not random:

- **sad → calm (47 clips)** — both low-arousal, low-energy. The model hears "quiet" and
  cannot tell apart "quiet and sad" from "quiet and calm".
- **happy → fearful (47 clips)** — both high-arousal. Same failure at the other end of
  the energy scale.
- **neutral → calm (22 clips)** — arguably the two most semantically similar labels in
  the whole set.

**The pattern: errors cluster by arousal (energy level), not randomly.** The model has
learned roughly *how activated* the speaker is, but struggles to distinguish emotions
that share an arousal level. That directly motivates the Phase 7 "arousal-first
two-stage classifier" experiment — predict high/low arousal first, then the specific
emotion within that branch.

In [22]:
best["per_class"].round(3)

,emotion,precision,recall,f1,support
0,neutral,0.423,0.229,0.297,96
1,calm,0.549,0.734,0.628,192
2,happy,0.414,0.427,0.421,192
3,sad,0.355,0.339,0.347,192
4,angry,0.655,0.594,0.623,192
5,fearful,0.452,0.562,0.501,192
6,disgust,0.527,0.562,0.544,192
7,surprised,0.765,0.526,0.623,192


**Neutral is the worst class by a wide margin** — recall 0.229, meaning we catch only 22
of 96 neutral clips. The Phase 1 audit predicted exactly this: neutral has half the
training data and the shortest speech after trimming.

This is *why we report macro-F1*. Accuracy is 0.515 but macro-F1 is 0.498 — the gap is
neutral (and sad) dragging the per-class average down. Accuracy alone would have hidden
it.

### Fairness and difficulty slices

`evaluate` always produces these, because a single headline number hides who the model
fails.

In [23]:
print("=== BY GENDER ===")
display(best["slices"]["gender"].round(4))

print("=== BY INTENSITY ===")
display(best["slices"]["intensity_label"].round(4))

=== BY GENDER ===


,level,n,accuracy,macro_f1
0,female,720,0.5542,0.5392
1,male,720,0.4750,0.4402


=== BY INTENSITY ===


,level,n,accuracy,macro_f1
0,normal,768,0.4349,0.4362
1,strong,672,0.6057,0.5337


**Gender: 7.9 pp gap** (female 55.4% vs male 47.5%), and ~10 pp on macro-F1. This is a
real fairness finding and it connects straight to the demographic-bias risk line in the
business case. A QA system that works measurably worse on male callers is a problem you
have to disclose, not bury.

**Intensity: 17.1 pp gap** (strong 60.6% vs normal 43.5%) — the largest slice effect we
measured. Strongly-performed emotion is much easier.

> **This one matters commercially.** Real contact-centre audio is overwhelmingly
> *normal* intensity — people do not perform emotions theatrically on support calls. So
> **43.5% is the more honest deployment estimate, not 51.5%.** RAVDESS is acted speech,
> and acted emotion is louder and clearer than the real thing.

*(Note: macro-F1 is `NaN` in the by-emotion slice by design. Each emotion subset has
only one true class, so averaging F1 over the seven absent classes produces a
meaningless number. We return `NaN` rather than something that looks like a score. This
was an actual bug we caught and fixed.)*

---
# Part 7 — Where we are, and what is next

## Honest summary of the numbers

| | Value |
|---|---|
| Best speaker-independent accuracy | **51.5%** (SVM-RBF, pooled) |
| Best speaker-independent macro-F1 | **49.8%** |
| Fold-to-fold spread | ±6.1 pp — quote it every time |
| Leakage gap | **13.75 pp** (26.7% relative inflation) |
| Random-chance baseline | 12.5% |
| Majority-class baseline | 13.3% |

**51.5% is ~3.9× the majority-class baseline.** It is a respectable floor for untuned
defaults on a hard 8-class problem — and it is *not* suspiciously high, which is itself
reassuring. If a speaker-independent number came back above 80%, the first assumption
should be a leak, not success.

## What we have NOT done yet

- **No hyperparameter tuning.** These are library defaults, on purpose.
- **No EDA figures** — Phase 2, restricted to training folds.
- **No feature comparison** — Phase 4 sweeps MFCC count, aggregation, and per-speaker
  normalisation.
- **No deep models** — Phase 6 (CNN on spectrograms, wav2vec2 embeddings).

## Adding your own experiment

Follow this and your results are automatically comparable with everyone else's:

```python
from ser.splits import get_split
from ser.features import FeatureConfig, extract, feature_columns
from ser.evaluate import evaluate_pooled
from ser.metadata import load

meta = load()
features = extract(FeatureConfig(n_mfcc=13))     # your config
cols = feature_columns(features, meta)
indexed = features.set_index("filename")

per_fold = []
for fold in range(5):
    train_files, test_files = get_split("speaker_independent", fold)
    # ... fit YOUR model on train_files, predict test_files ...
    per_fold.append((y_true, y_pred, indexed.loc[test_files].reset_index()))

evaluate_pooled(per_fold, "speaker_independent",
                model="my-model", feature_config="mfcc13")
```

**Four things to never do:**

1. Never build your own splits — always `get_split`.
2. Never fit a scaler outside a `Pipeline` (or outside the training fold).
3. Never compute accuracy yourself — always go through `evaluate`.
4. Never report a random-stratified number without the speaker-independent one beside it.

## Reference documents

| File | Contents |
|---|---|
| `CLAUDE.md` | Project rules, data facts, conventions |
| `PLAN.md` | Phase checklist and gates |
| `docs/data_audit.md` | Full audit and the four preprocessing decisions |
| `docs/splits.md` | The frozen fold contract |
| `results/results.csv` | Every run anyone has logged |

Run the test suite any time with `pytest` — currently **89 tests**, covering every
invariant above.

---
# Where to go next

This notebook covered construction: the rules, the data, the audit, the folds,
the harness, and the first baseline.

**For the final results — the 63.9% headline, the 92.7% actor-identification
finding, the fairness slices and the two experiments that failed — open
`01_results_and_findings.ipynb`.**

| Document | Contents |
|---|---|
| `docs/data_audit.md` | Audit + four preprocessing decisions |
| `docs/splits.md` | The frozen fold contract |
| `docs/feature_findings.md` | Phase 4 ablation, 19 configs |
| `docs/baseline_findings.md` | Phase 5, seven models + tuning |
| `docs/novelty_findings.md` | Phase 7, six experiments |
| `docs/DECK.md` | Presentation content and speaker notes |